# Math 189 Final Project — Data Cleaning & Preparation

**Research Question:**
> For San Francisco from 2023 to 2025, did the expansion of Waymo's driverless ride-hailing service correspond with changes in taxi activity, public transit ridership, and the broader transportation market?

**Goal of this notebook:**

This notebook loads, cleans, and merges four raw transportation datasets into a single monthly panel dataset ready for exploratory data analysis (EDA) and regression modeling.

The four data sources are:
1. **Waymo / AV activity** — California statewide monthly robotaxi passenger miles (CPUC via Our World in Data)
2. **SF Taxi trips** — Monthly SF taxi trip counts aggregated from DataSF
3. **Muni ridership** — Monthly total boardings from SFMTA
4. **SFO ground transportation** — Monthly airport ground trip counts by permit type (optional extension)

The output of this notebook is a single clean CSV:
`data/02-processed/final_transportation_monthly.csv`

---

**Key Waymo expansion dates used in this analysis:**
- **August 2023** — Waymo opens commercial driverless service to the general public in SF
- **June 2024** — Waymo One opens to everyone in SF (full unrestricted public launch)

---
## 1. Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

print('All packages loaded successfully.')

---
## 2. Load Raw Datasets

We load each dataset and immediately run basic diagnostics (`.head()`, `.shape`, `.info()`, missing values) to understand what we are working with before cleaning.

### 2.1 Waymo / AV Activity Dataset

This dataset comes from the California Public Utilities Commission (CPUC) via Our World in Data. It reports monthly passenger miles traveled by robotaxis in California. We load it directly from the Our World in Data URL, with a local fallback.

In [ ]:
WAYMO_URL = (
    'https://ourworldindata.org/grapher/passenger-miles-traveled-self-driving-taxis.csv'
    '?v=1&csvType=full&useColumnShortNames=false'
)
WAYMO_LOCAL = 'data/passenger-miles-traveled-self-driving-taxis.csv'

try:
    df_waymo_raw = pd.read_csv(WAYMO_URL)
    print('Loaded Waymo data from URL.')
except Exception:
    df_waymo_raw = pd.read_csv(WAYMO_LOCAL)
    print('Loaded Waymo data from local file.')

print(f'Shape: {df_waymo_raw.shape}')
df_waymo_raw.head()

In [ ]:
df_waymo_raw.info()

In [ ]:
print('Missing values:')
print(df_waymo_raw.isnull().sum())

### 2.2 SF Taxi Trips Dataset

The full DataSF Taxi Trips dataset contains over 4 million individual trip records. Loading all of it into memory would be impractical, so we use the Socrata API's `$select` and `$group` parameters to request pre-aggregated monthly counts directly from the server. This returns one row per month instead of millions of trip-level rows.

We also have a local pre-aggregated file as a fallback.

In [ ]:
TAXI_API = (
    'https://data.sfgov.org/resource/m8hk-2ipk.csv'
    '?$select=date_trunc_ym(start_time_local)%20as%20month,'
    'count(*)%20as%20taxi_trips,'
    'sum(total_fare_amount)%20as%20taxi_total_fare'
    '&$group=month'
    '&$order=month'
    '&$limit=100'
)
TAXI_LOCAL = 'data/sf_taxi_monthly.csv'

try:
    df_taxi_raw = pd.read_csv(TAXI_API)
    print('Loaded taxi data from API.')
except Exception:
    df_taxi_raw = pd.read_csv(TAXI_LOCAL)
    print('Loaded taxi data from local file (no fare totals).')

print(f'Shape: {df_taxi_raw.shape}')
df_taxi_raw.head(10)

In [ ]:
df_taxi_raw.info()

In [ ]:
print('Missing values:')
print(df_taxi_raw.isnull().sum())

### 2.3 Muni Ridership Dataset

This dataset comes from SFMTA's public Tableau dashboard. Each calendar month is represented by three rows — one for monthly total boardings, one for the 2019 baseline boardings, and one for the recovery percentage. We will reshape this into one row per month during cleaning.

In [ ]:
MUNI_URL = (
    'https://transtat.sfmta.com/t/public/views/SystemwideRidershipRecovery/'
    'MonthlySystemwideRecoveryAccessibleTable.csv'
)
MUNI_LOCAL = 'data/muni_ridership_monthly.csv'

try:
    df_muni_raw = pd.read_csv(
        MUNI_URL,
        headers={'User-Agent': 'Mozilla/5.0'}
    )
    print('Loaded Muni data from URL.')
except Exception:
    df_muni_raw = pd.read_csv(MUNI_LOCAL)
    print('Loaded Muni data from local file.')

print(f'Shape: {df_muni_raw.shape}')
df_muni_raw.head(9)

In [ ]:
df_muni_raw.info()

In [ ]:
print('Missing values:')
print(df_muni_raw.isnull().sum())

print('\nUnique measure names:')
print(df_muni_raw['Measure Names'].unique())

### 2.4 SFO Ground Transportation Dataset (Optional)

This dataset from DataSF contains monthly ground transportation trip counts at San Francisco International Airport, broken down by permit type (e.g., TNCs, taxis, shared ride vans). We include it as an optional extension that provides a narrower, well-defined transportation setting in case city-wide comparisons are difficult to isolate.

We use the Socrata API to load a filtered and aggregated version.

In [ ]:
SFO_API = (
    'https://data.sfgov.org/resource/e3x5-4br3.csv'
    '?$limit=50000'
)

try:
    df_sfo_raw = pd.read_csv(SFO_API)
    print(f'Loaded SFO data from API. Shape: {df_sfo_raw.shape}')
    df_sfo_raw.head()
except Exception as e:
    df_sfo_raw = None
    print(f'Could not load SFO data: {e}')

In [ ]:
if df_sfo_raw is not None:
    print(f'Shape: {df_sfo_raw.shape}')
    print('\nColumns:', df_sfo_raw.columns.tolist())
    print('\nMissing values:')
    print(df_sfo_raw.isnull().sum())
    df_sfo_raw.head()

---
## 3. Clean the Waymo Dataset

**What we do here:**
- Parse the `Day` column as a proper datetime
- Convert the end-of-month date to a year-month period for merging
- Filter to the 2023–2025 study window
- Rename columns clearly
- Inspect the result

In [ ]:
df_waymo = df_waymo_raw.copy()

# Rename columns to something clear and consistent
df_waymo.columns = df_waymo.columns.str.strip()
print('Original columns:', df_waymo.columns.tolist())

In [ ]:
# The date column is named 'Day' and contains end-of-month dates (e.g. 2023-01-31)
# We convert it to datetime, then to a monthly period for clean merging
df_waymo['Day'] = pd.to_datetime(df_waymo['Day'])
df_waymo['month'] = df_waymo['Day'].dt.to_period('M')

# Rename the passenger miles column
waymo_miles_col = [c for c in df_waymo.columns if 'distance' in c.lower() or 'miles' in c.lower() or 'passenger' in c.lower()][0]
print(f'Identified passenger miles column: "{waymo_miles_col}"')
df_waymo = df_waymo.rename(columns={waymo_miles_col: 'waymo_passenger_miles'})

# Keep only California rows (the dataset may contain multiple entities)
if 'Entity' in df_waymo.columns:
    df_waymo = df_waymo[df_waymo['Entity'] == 'California']

# Keep only the columns we need
df_waymo = df_waymo[['month', 'waymo_passenger_miles']].copy()

# Filter to the study window
df_waymo = df_waymo[
    (df_waymo['month'] >= '2023-01') &
    (df_waymo['month'] <= '2025-12')
].reset_index(drop=True)

print(f'\nRows after filtering to 2023–2025: {len(df_waymo)}')
print(f'Date range: {df_waymo["month"].min()} to {df_waymo["month"].max()}')
df_waymo.head()

In [ ]:
# Sanity check: one row per month, no missing values
print(f'Duplicate months: {df_waymo["month"].duplicated().sum()}')
print(f'Missing values:\n{df_waymo.isnull().sum()}')
df_waymo.describe()

---
## 4. Clean the Taxi Dataset

**What we do here:**
- Parse the `month` column as a datetime and convert to a monthly period
- Cast trip and fare columns to numeric types
- Filter to the 2023–2025 study window
- Compute average fare per trip as an additional variable
- Note the known data gap: the DataSF taxi dataset ends in May 2024

> **Important limitation:** No public SF taxi data exists after May 2024. This means the taxi series covers only January 2023 – May 2024 within our study window. We will flag these missing months in the merged dataset.

In [ ]:
df_taxi = df_taxi_raw.copy()

# Parse the month column — handles both 'YYYY-MM-DDTHH:MM:SS' (API) and 'YYYY-MM' (local)
df_taxi['month'] = pd.to_datetime(df_taxi['month']).dt.to_period('M')

# Convert trip count and fare to numeric (API may return strings)
df_taxi['taxi_trips'] = pd.to_numeric(df_taxi['taxi_trips'], errors='coerce')

if 'taxi_total_fare' in df_taxi.columns:
    df_taxi['taxi_total_fare'] = pd.to_numeric(df_taxi['taxi_total_fare'], errors='coerce')
    df_taxi['taxi_avg_fare'] = df_taxi['taxi_total_fare'] / df_taxi['taxi_trips']
else:
    print('Note: taxi_total_fare not in dataset (local file). Average fare cannot be computed.')
    df_taxi['taxi_total_fare'] = np.nan
    df_taxi['taxi_avg_fare'] = np.nan

# Filter to the study window
df_taxi = df_taxi[
    (df_taxi['month'] >= '2023-01') &
    (df_taxi['month'] <= '2025-12')
].reset_index(drop=True)

print(f'Rows after filtering to 2023–2025: {len(df_taxi)}')
print(f'Date range: {df_taxi["month"].min()} to {df_taxi["month"].max()}')
print('\nNote: taxi data ends May 2024. Months after that will be NaN in the merged dataset.')
df_taxi.head(10)

In [ ]:
# Sanity check
print(f'Duplicate months: {df_taxi["month"].duplicated().sum()}')
print(f'Missing values:\n{df_taxi.isnull().sum()}')
df_taxi.describe()

---
## 5. Clean the Muni Ridership Dataset

**What we do here:**
- The raw file has three rows per month (one per measure type). We pivot it so each month becomes one row.
- Parse the month string (e.g., `"January 2023"`) to a monthly period.
- Extract the key variable: `muni_monthly_boardings` (total monthly boardings).
- Extract `muni_recovery_rate` as an optional supplementary variable.
- Filter to 2023–2025.

In [ ]:
df_muni = df_muni_raw.copy()

# Standardize column names
df_muni.columns = ['measure', 'month_str', 'value']

# Remove commas from numbers stored as strings (e.g., '2,555,000' -> 2555000)
df_muni['value'] = (
    df_muni['value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

# Parse the month string to a Period
df_muni['month'] = pd.to_datetime(df_muni['month_str'], format='%B %Y').dt.to_period('M')

print('Unique measure types:')
print(df_muni['measure'].unique())
df_muni.head(9)

In [ ]:
# Pivot: one row per month, one column per measure
df_muni_wide = df_muni.pivot_table(
    index='month',
    columns='measure',
    values='value',
    aggfunc='first'
).reset_index()

# Flatten column names
df_muni_wide.columns.name = None

# Rename to clean variable names
# IMPORTANT: check 'Baseline' BEFORE 'Monthly Total Boardings' because
# "Baseline Monthly Total Boardings (accessible copy)" contains both substrings.
# Checking the more specific condition first prevents a duplicate column name.
rename_map = {}
for col in df_muni_wide.columns:
    if 'Baseline' in col:
        rename_map[col] = 'muni_baseline_boardings'
    elif 'Monthly Total Boardings' in col:
        rename_map[col] = 'muni_monthly_boardings'
    elif 'Recovery' in col:
        rename_map[col] = 'muni_recovery_rate'

df_muni_wide = df_muni_wide.rename(columns=rename_map)

# Filter to the study window
df_muni_wide = df_muni_wide[
    (df_muni_wide['month'] >= '2023-01') &
    (df_muni_wide['month'] <= '2025-12')
].reset_index(drop=True)

print(f'Rows after filtering to 2023–2025: {len(df_muni_wide)}')
print(f'Date range: {df_muni_wide["month"].min()} to {df_muni_wide["month"].max()}')
print(f'Columns: {df_muni_wide.columns.tolist()}')
df_muni_wide.head()

In [ ]:
# Sanity check
print(f'Duplicate months: {df_muni_wide["month"].duplicated().sum()}')
print(f'Missing values:\n{df_muni_wide.isnull().sum()}')
df_muni_wide.describe()

---
## 6. Clean the SFO Ground Transportation Dataset (Optional)

SFO ground transportation data records monthly trip counts at San Francisco International Airport by permit type (e.g., taxis, TNCs, shared ride vans). This is an optional extension. We include it here because:
1. It provides a focused transportation setting where Waymo, taxis, and TNCs directly compete.
2. SFO data is consistently tracked and publicly available.

**What we do here:**
- Inspect the columns to understand the structure.
- Parse and standardize the date column.
- Filter to 2023–2025.
- Aggregate monthly total trips and, if possible, separate TNC vs. taxi trips.

In [ ]:
if df_sfo_raw is not None:
    df_sfo = df_sfo_raw.copy()

    print('Columns:', df_sfo.columns.tolist())
    print('\nShape:', df_sfo.shape)
    df_sfo.head()
else:
    print('SFO dataset was not loaded. Skipping this section.')

In [ ]:
if df_sfo_raw is not None:
    # Identify the date and trip count columns
    # Column names may vary — inspect and adapt as needed
    date_col = [c for c in df_sfo.columns if 'date' in c.lower() or 'month' in c.lower() or 'period' in c.lower()][0]
    trip_col = [c for c in df_sfo.columns if 'trip' in c.lower() or 'count' in c.lower()]
    permit_col = [c for c in df_sfo.columns if 'permit' in c.lower() or 'type' in c.lower()]

    print(f'Date column identified: {date_col}')
    print(f'Trip count columns: {trip_col}')
    print(f'Permit type columns: {permit_col}')

    if permit_col:
        print('\nUnique permit types:')
        print(df_sfo[permit_col[0]].value_counts())

In [ ]:
if df_sfo_raw is not None:
    # Parse date and aggregate to monthly level
    df_sfo[date_col] = pd.to_datetime(df_sfo[date_col], errors='coerce')
    df_sfo['month'] = df_sfo[date_col].dt.to_period('M')

    # Use the first identified trip count column
    trip_count_col = trip_col[0] if trip_col else None

    if trip_count_col:
        df_sfo[trip_count_col] = pd.to_numeric(df_sfo[trip_count_col], errors='coerce')

        # Aggregate: total trips per month
        df_sfo_monthly = (
            df_sfo.groupby('month')[trip_count_col]
            .sum()
            .reset_index()
            .rename(columns={trip_count_col: 'sfo_ground_trips'})
        )

        # Filter to study window
        df_sfo_monthly = df_sfo_monthly[
            (df_sfo_monthly['month'] >= '2023-01') &
            (df_sfo_monthly['month'] <= '2025-12')
        ].reset_index(drop=True)

        print(f'SFO monthly rows: {len(df_sfo_monthly)}')
        print(f'Date range: {df_sfo_monthly["month"].min()} to {df_sfo_monthly["month"].max()}')
        df_sfo_monthly.head()
    else:
        df_sfo_monthly = None
        print('Could not identify a trip count column in the SFO dataset.')
else:
    df_sfo_monthly = None

---
## 7. Save Cleaned Intermediate Datasets

Before merging, we save each cleaned dataset to `data/02-processed/`. This is good practice — if something goes wrong in the merge step, we do not need to re-run the cleaning steps.

In [ ]:
import os
os.makedirs('data/02-processed', exist_ok=True)

# Save each cleaned intermediate file
df_waymo.to_csv('data/02-processed/waymo_clean.csv', index=False)
df_taxi.to_csv('data/02-processed/taxi_clean.csv', index=False)
df_muni_wide.to_csv('data/02-processed/muni_clean.csv', index=False)

if df_sfo_monthly is not None:
    df_sfo_monthly.to_csv('data/02-processed/sfo_clean.csv', index=False)
    print('Saved: waymo_clean.csv, taxi_clean.csv, muni_clean.csv, sfo_clean.csv')
else:
    print('Saved: waymo_clean.csv, taxi_clean.csv, muni_clean.csv')

---
## 8. Merge Datasets

We now merge all cleaned datasets into a single monthly panel on the `month` column.

**Merge strategy:**
We use a **left join starting from Waymo** as the base, because Waymo activity is our primary independent variable and has the broadest coverage (through December 2025). Every month with Waymo data will appear in the final dataset. Months where other datasets have no data (e.g., taxi data after May 2024) will appear as `NaN`, which is the honest representation of a data gap.

We then inspect the merged result for missing months and missing values.

In [ ]:
# Start with the Waymo series as the base (broadest coverage)
df_merged = df_waymo.copy()

# Left join: taxi trips
df_merged = df_merged.merge(df_taxi[['month', 'taxi_trips', 'taxi_total_fare', 'taxi_avg_fare']],
                             on='month', how='left')

# Left join: Muni ridership
muni_cols = ['month', 'muni_monthly_boardings', 'muni_recovery_rate']
muni_cols = [c for c in muni_cols if c in df_muni_wide.columns]
df_merged = df_merged.merge(df_muni_wide[muni_cols], on='month', how='left')

# Left join: SFO ground trips (optional)
if df_sfo_monthly is not None:
    df_merged = df_merged.merge(df_sfo_monthly, on='month', how='left')

# Sort chronologically
df_merged = df_merged.sort_values('month').reset_index(drop=True)

# Drop any duplicate columns (keeps the first occurrence)
df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]

print(f'Merged dataset shape: {df_merged.shape}')
print(f'Columns: {df_merged.columns.tolist()}')
print(f'Date range: {df_merged["month"].min()} to {df_merged["month"].max()}')
df_merged.head()

In [ ]:
# Check for missing values after merge
print('Missing values per column:')
missing = df_merged.isnull().sum()
missing_pct = (df_merged.isnull().sum() / len(df_merged) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0])

In [ ]:
# Confirm one row per month
print(f'Total rows: {len(df_merged)}')
print(f'Unique months: {df_merged["month"].nunique()}')
print(f'Duplicate months: {df_merged["month"].duplicated().sum()}')

In [ ]:
# Check for any gaps in the monthly sequence (months that should exist but don't)
expected_months = pd.period_range(
    start=df_merged['month'].min(),
    end=df_merged['month'].max(),
    freq='M'
)
actual_months = set(df_merged['month'])
missing_months = [m for m in expected_months if m not in actual_months]

if missing_months:
    print(f'WARNING: {len(missing_months)} month(s) are missing from the sequence:')
    print(missing_months)
else:
    print('No gaps in the monthly sequence. Every month is represented.')

---
## 9. Feature Engineering

We create several new variables that will be needed for EDA and regression analysis.

| Variable | Description |
|---|---|
| `post_waymo_public_launch` | 1 if month ≥ August 2023 (Waymo opens to public) |
| `post_waymo_open_to_all` | 1 if month ≥ June 2024 (Waymo fully open, no waitlist) |
| `time_trend` | Integer index starting at 0 (Jan 2023 = 0), captures secular trend |
| `waymo_pct_change` | Month-over-month % change in Waymo passenger miles |
| `taxi_pct_change` | Month-over-month % change in taxi trips |
| `muni_pct_change` | Month-over-month % change in Muni boardings |
| `log_waymo_passenger_miles` | Natural log of Waymo miles (useful for regression) |
| `log_taxi_trips` | Natural log of taxi trips |
| `log_muni_monthly_boardings` | Natural log of Muni boardings |

In [ ]:
df = df_merged.copy()

# --- Waymo expansion indicator variables ---
df['post_waymo_public_launch'] = (df['month'] >= '2023-08').astype(int)
df['post_waymo_open_to_all']   = (df['month'] >= '2024-06').astype(int)

# --- Time trend: 0 = January 2023 ---
start_month = pd.Period('2023-01', freq='M')
df['time_trend'] = (df['month'] - start_month).apply(lambda x: x.n)

print('Expansion indicators created.')
print(df[['month', 'post_waymo_public_launch', 'post_waymo_open_to_all', 'time_trend']].head(12))

In [ ]:

df['waymo_pct_change'] = df['waymo_passenger_miles'].pct_change() * 100
df['taxi_pct_change']  = df['taxi_trips'].pct_change() * 100
df['muni_pct_change']  = df['muni_monthly_boardings'].pct_change() * 100

print('Percent change variables created.')
df[['month', 'waymo_pct_change', 'taxi_pct_change', 'muni_pct_change']].head(6)

In [ ]:
# --- Log-transformed variables ---
# Log transforms are useful in regression when variables span multiple orders of magnitude.
# np.log1p(x) = log(1 + x), which safely handles zero values.
df['log_waymo_passenger_miles']  = np.log1p(df['waymo_passenger_miles'])
df['log_taxi_trips']             = np.log1p(df['taxi_trips'])
df['log_muni_monthly_boardings'] = np.log1p(df['muni_monthly_boardings'])

print('Log-transformed variables created.')
df[['month', 'log_waymo_passenger_miles', 'log_taxi_trips', 'log_muni_monthly_boardings']].head(6)

In [ ]:
# Final column list
print('All columns in final dataset:')
for c in df.columns:
    print(f'  {c}')

---
## 10. Final Data Checks

Before saving, we perform a series of final checks to confirm the dataset is well-formed and ready for analysis.

In [ ]:
print('=== SHAPE ===')
print(f'{df.shape[0]} rows x {df.shape[1]} columns')

print('\n=== DATE RANGE ===')
print(f'{df["month"].min()} to {df["month"].max()}')

print('\n=== ONE ROW PER MONTH ===')
print(f'Duplicate months: {df["month"].duplicated().sum()}')

In [ ]:
df.head(6)

In [ ]:
df.tail(6)

In [ ]:
df.info()

In [ ]:
# Summary statistics for the main variables
main_vars = [
    'waymo_passenger_miles', 'taxi_trips', 'muni_monthly_boardings',
    'post_waymo_public_launch', 'post_waymo_open_to_all', 'time_trend'
]
df[main_vars].describe().round(2)

In [ ]:
# Missing value summary
print('Missing values in final dataset:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0])

In [ ]:
# Quick time-series overview plot of the three main variables
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

month_dt = df['month'].dt.to_timestamp()

# Waymo passenger miles
axes[0].plot(month_dt, df['waymo_passenger_miles'], marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Waymo Passenger Miles (California Statewide)')
axes[0].set_ylabel('Passenger Miles')

# Taxi trips
axes[1].plot(month_dt, df['taxi_trips'], marker='o', linewidth=2, color='tomato')
axes[1].set_title('SF Monthly Taxi Trips')
axes[1].set_ylabel('Trips')

# Muni boardings
axes[2].plot(month_dt, df['muni_monthly_boardings'], marker='o', linewidth=2, color='seagreen')
axes[2].set_title('Muni Monthly Total Boardings')
axes[2].set_ylabel('Boardings')

# Add Waymo expansion markers to all panels
for ax in axes:
    ax.axvline(pd.Timestamp('2023-08-01'), color='orange', linestyle='--', linewidth=1.5,
               label='Aug 2023: Waymo public launch')
    ax.axvline(pd.Timestamp('2024-06-01'), color='purple', linestyle='--', linewidth=1.5,
               label='Jun 2024: Waymo open to all')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

axes[0].legend(fontsize=8)
plt.xticks(rotation=45)
plt.suptitle('Monthly Transportation Activity in San Francisco (2023–2025)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('data/02-processed/overview_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Overview plot saved.')

---
## 11. Save the Final Dataset

We save the final merged and feature-engineered dataset. Because `pandas.Period` objects are not directly serializable to CSV, we first convert the `month` column to a string in `YYYY-MM` format before saving.

In [ ]:
df_save = df.copy()

# Convert Period to string for CSV compatibility
df_save['month'] = df_save['month'].astype(str)

output_path = 'data/02-processed/final_transportation_monthly.csv'
df_save.to_csv(output_path, index=False)

print(f'Final dataset saved to: {output_path}')
print(f'Shape: {df_save.shape}')
print(f'Columns: {df_save.columns.tolist()}')

---
## Summary

This notebook loaded, cleaned, and merged four transportation datasets into a single monthly panel dataset. Here is what was done:

| Step | Action |
|---|---|
| Waymo data | Loaded from OWID/CPUC, filtered to 2023–2025, renamed columns |
| Taxi data | Loaded pre-aggregated monthly counts from DataSF API or local file |
| Muni data | Loaded from SFMTA Tableau, pivoted from long to wide format |
| SFO data | Loaded and aggregated by month (optional) |
| Merge | Left join on `month`, Waymo as the base |
| Features | Added expansion dummies, time trend, pct change, log transforms |
| Output | Saved to `data/02-processed/final_transportation_monthly.csv` |

**Known limitations carried into the final dataset:**
- `taxi_trips` and fare variables are `NaN` after May 2024 (no public data exists)
- `waymo_passenger_miles` is California statewide, not SF-only
- BART ridership was not included (available separately in `data/bart_monthly_exits.csv` if needed)

**Next step:** Open `02-eda.ipynb` for exploratory data analysis.